1. Load image_manifest.csv and select images where split == "validation".
2. Use the finalized prompt and JSON Schema.
3. Save each Gemini prediction individually, with API error handling and automatic skipping of previously processed images.
4. Perform the same quantitative evaluations on all 61 validation images.
5. Generate final comparison tables for follicle-like region counts, sizes, and spatial agreement.

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import json

drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/ovarian_ultrasound_vlm"
)

MANIFEST_PATH = PROJECT_DIR / "image_manifest.csv"

manifest = pd.read_csv(MANIFEST_PATH)

assert len(manifest) == 301
assert manifest["case_id"].is_unique

print("Total dataset images:", len(manifest))

Mount Google Drive and load the dataset manifest for the main evaluation

In [ ]:
validation = (
    manifest[manifest["split"] == "validation"]
    .sort_values("case_id")
    .reset_index(drop=True)
)

assert len(validation) == 61
assert validation["case_id"].is_unique

# Make sure no pilot images are included
assert not validation["is_pilot"].astype(str).str.lower().eq("true").any()

# Verify that all images and pseudo-labels exist
for _, row in validation.iterrows():
    assert Path(row["image_path"]).exists()
    assert Path(row["pseudo_label_path"]).exists()

print("Validation images:", len(validation))

print("\nCategory distribution:")
print(validation["category"].value_counts().sort_index())

display(validation[["case_id", "category", "split"]].head(10))

Select the 61 validation cases from the original nnU-Net fold 0